# PNPL decoder: Kaggle CLI entry
Enable Internet and a GPU. Set REPO_COMMIT to the full 40-character commit containing this implementation after it is published. The notebook intentionally refuses an unset pin; it does not install a moving branch. Set a preparation disk budget appropriate to your actual disk. No final-test preparation, prediction or submission is launched here.

For resume, copy the previous trusted run's latest.pt (and best.pt for older runs) into OUTPUT first, set CONFIG to the same configuration and TOTAL_EPOCHS to the original total epoch override (not remaining epochs), and set RESUME=True. Leave CONFIG=None for the installed baseline and TOTAL_EPOCHS=None to use its configured epoch budget. Kaggle working storage is session-local: persist/download outputs before the session ends.

In [ ]:
import re
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/yjoechuen/pnpl-decoder.git'
REPO_COMMIT = ''  # Required: published commit containing this implementation.
DISK_BUDGET_GIB = 15
HEADROOM_GIB = 5
RESUME = False
CONFIG = None  # Custom config path shared by preparation and training.
TOTAL_EPOCHS = None  # Original total --epochs override for resume, if used.
DATA = Path('/kaggle/working/pnpl-decoder/data')
OUTPUT = Path('/kaggle/working/pnpl-decoder/artifacts/baseline')
if not re.fullmatch(r'[0-9a-f]{40}', REPO_COMMIT):
    raise ValueError('Set REPO_COMMIT to a published full commit SHA, never a branch name')
subprocess.run([sys.executable, '-m', 'pip', 'install',
                f'pnpl-decoder @ git+{REPO_URL}@{REPO_COMMIT}'], check=True)


In [ ]:
from decoder import default_config_path
config_args = ['--config', str(CONFIG if CONFIG is not None else default_config_path())]
prepare = ['pnpl-decoder', '--profile', 'kaggle', 'prepare',
           '--data', str(DATA), '--index-only',
           '--disk-budget-gib', str(DISK_BUDGET_GIB), '--headroom-gib', str(HEADROOM_GIB)] + config_args
subprocess.run(prepare + ['--preflight'], check=True)
subprocess.run(prepare, check=True)
train = ['pnpl-decoder', '--profile', 'kaggle', 'train', '--data', str(DATA),
         '--output', str(OUTPUT), '--data-mode', 'h5', '--device', 'cuda'] + config_args
if TOTAL_EPOCHS is not None:
    train.extend(['--epochs', str(TOTAL_EPOCHS)])
if RESUME:
    train.append('--resume')
subprocess.run(train, check=True)
